# Study 812 — Corwin-Schultz Spread — the teardown

The per-leg books, the Newey-West spread *t*, the pooled Welch book test, the 1,000-permutation placebo, the two-era robustness cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_days': 4125, 'fingerprint': '357fd262912f', 'median_spread_bps': 12.7, 'spread_bps': 4.45, 't_nw': 3.24, 't_1s': 3.06, 'long_bps': 10.64, 'short_bps': 6.19, 'welch_t': 1.62, 'gross_sharpe': 0.76, 'placebo_obs': 4.45, 'placebo_mean': -0.019, 'placebo_sd': 0.916, 'placebo_p': 0.0, 'placebo_sigma': 4.88, 'placebo_draws': 1000, 'era_early_bps': 3.48, 'era_early_t': 2.03, 'era_early_n': 1991, 'era_late_bps': 5.35, 'era_late_t': 2.52, 'era_late_n': 2134, 'timer_1_gross': 4.45, 'timer_1_cost': 2.14, 'timer_1_net': 2.31, 'timer_1_t': 1.59, 'timer_1_sharpe': 0.39, 'timer_1_ann': 5.8, 'timer_5_gross': 4.45, 'timer_5_cost': 10.14, 'timer_5_net': -5.69, 'timer_5_t': -3.91, 'timer_5_sharpe': -0.97, 'timer_5_ann': -14.3, 'null_mean_t': -0.17, 'null_sd_t': 0.79, 'null_fire': 1, 'planted_t': 10.09, 'planted_welch': 10.22}

## The estimator itself

Daily `S = 2(e^α−1)/(1+e^α)` from consecutive 2-day highs/lows (negatives floored at 0), averaged over a trailing month. The median mega-cap estimate is a sensible effective spread — a sanity check before the sort.

In [2]:
print(f"median daily CS spread across names : ~{R['median_spread_bps']:.1f} bps")
print(f"panel Close fingerprint             : {R['fingerprint']}  (as-of {R['end']})")

median daily CS spread across names : ~12.7 bps
panel Close fingerprint             : 357fd262912f  (as-of 2026-06-30)


## The headline — long-high-spread / short-low-spread

Daily equal-weight top-30% (illiquid) minus bottom-30% (liquid) estimated-spread sort.

In [3]:
print(f"spread        : {R['spread_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : high-spread {R['long_bps']:+.2f} vs low-spread {R['short_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:.2f} (before cost)")

spread        : +4.45 bps/day  NW(10) t = +3.24  one-sample t = +3.06
books         : high-spread +10.64 vs low-spread +6.19 bps (Welch t = +1.62)
gross Sharpe  : 0.76 (before cost)


## Placebo — column-permute the forward returns (1,000 permutations)

In [4]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> p = {R['placebo_p']:.5f}")
print(f"observed is {R['placebo_sigma']:+.2f} sigma into the RIGHT tail of the null")

observed +4.45 bps vs placebo mean -0.019 (sd 0.916) -> p = 0.00000
observed is +4.88 sigma into the RIGHT tail of the null


## Robustness — two eras (split 2018-01-01)

In [5]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")

2010-2017 (n=1991): +3.48 bps  NW t = +2.03
2018-2026 (n=2134): +5.35 bps  NW t = +2.52


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per day on the long-short book; short pays 50 bps/yr borrow. (The flat charge is *generous* — the illiquid long leg is where real spreads are widest.)

In [6]:
for tag,g,c,n,t,sh,an in [
    ('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t'],R['timer_1_sharpe'],R['timer_1_ann']),
    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'],R['timer_5_sharpe'],R['timer_5_ann'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day "
          f"(cost {c:.2f}/day, t={t:+.2f}, Sharpe {sh:+.2f}, ~{an:+.1f}%/yr)")

 1 bp one-way: gross +4.45 -> net +2.31 bps/day (cost 2.14/day, t=+1.59, Sharpe +0.39, ~+5.8%/yr)
5 bps one-way: gross +4.45 -> net -5.69 bps/day (cost 10.14/day, t=-3.91, Sharpe -0.97, ~-14.3%/yr)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted premium.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from corwin_schultz import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=812+s, n_assets=40, n_days=1200))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.08, seed=812, n_assets=40, n_days=1500))
print(f"planted (edge=0.08): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (edge=0), 8 seeds: NW t mean -0.01 (sd 0.65), |t|>=2 in 0/8


planted (edge=0.08): NW t = +10.09, Welch t = +10.22


## Verdict

- **Signal — Real.** The Corwin-Schultz illiquidity premium **replicates with the correct sign** on 50 liquid US mega-caps: long high-spread / short low-spread is **+4.45 bps/day** (NW *t* = **+3.24**), significant in both eras (*t* = +2.03 / +2.52), +4.88σ into the right tail of a 1,000-permutation placebo. The 20-seed synthetic control recovers a *planted* premium cleanly (*t* = +10.09, fires on 1/20 nulls). A rare green — it survives even where an illiquidity effect should be weakest. Survivorship biases the magnitude upward.
- **Tradability — Fragile.** The gross premium is real but lives inside its own cost band: at 1 bp one-way net **+2.31 bps/day** but *t* only +1.59; at 5 bps **-5.69 bps/day** (*t* = -3.91). The long leg is the illiquid names — you pay the very spread the premium is compensating.